In [4]:
import json
import os
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
 
import pandas as pd
import requests
from dotenv import load_dotenv
 
BASE_URL = "https://api.swedavia.se/flightinfo/v2"
RAW_DIR = Path("data/raw/arrivals")
 
# Swedavia airports (section 1.3.1 in the docs)
AIRPORTS = ["ARN", "BMA", "GOT", "MMX", "LLA", "UME", "OSD", "VBY", "RNB", "KRN"]
 
load_dotenv()
API_KEY = os.getenv("SWEDAVIA_API_KEY")
if API_KEY is None:
    raise RuntimeError("SWEDAVIA_API_KEY not found, check your .env")
 
HEADERS = {
    "Accept": "application/json",
    "Ocp-Apim-Subscription-Key": API_KEY,
}
 
 
def get_arrivals(airport: str, flight_date: date) -> dict:
    """Fetch all arrivals for one airport on one date (UTC)."""
    url = f"{BASE_URL}/{airport}/arrivals/{flight_date.isoformat()}"
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()
 
 
def save_raw(data: dict, airport: str, flight_date: date) -> Path:
    """Save the untouched API response as JSON (bronze layer)."""
    path = RAW_DIR / airport / f"{flight_date.isoformat()}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return path
 
 
def flights_to_df(data: dict, airport: str) -> pd.DataFrame:
    """Flatten the nested flights list into a DataFrame."""
    # Docs use PascalCase, the JSON is usually camelCase, so handle both
    flights = data.get("flights") or data.get("Flights") or []
    df = pd.json_normalize(flights)
    df["airport"] = airport
    return df
 
 
def fetch_all_arrivals(flight_date: date, airports: list[str] = AIRPORTS) -> pd.DataFrame:
    """Fetch, save and combine arrivals for all airports on one date."""
    frames = []
    for airport in airports:
        try:
            data = get_arrivals(airport, flight_date)
        except requests.HTTPError as e:
            print(f"{airport}: failed ({e})")
            continue
 
        save_raw(data, airport, flight_date)
        df = flights_to_df(data, airport)
        print(f"{airport}: {len(df)} flights")
        frames.append(df)
 
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)
 
 
def two_days_ago_utc() -> date:
    #TODO: FIX so this so it gets 3 days instead of two
    """The API expects dates in UTC, so don't use date.today()."""
    return datetime.now(timezone.utc).date() - timedelta(days=2)
 
df_arrivals = fetch_all_arrivals(two_days_ago_utc())
print(df_arrivals["airport"])

ARN: 349 flights
BMA: 23 flights
GOT: 77 flights
MMX: 22 flights
LLA: 15 flights
UME: 15 flights
OSD: 6 flights
VBY: 12 flights
RNB: 6 flights
KRN: 2 flights
0      ARN
1      ARN
2      ARN
3      ARN
4      ARN
      ... 
522    RNB
523    RNB
524    RNB
525    KRN
526    KRN
Name: airport, Length: 527, dtype: str


In [5]:
df_arrivals

,flightId,departureAirportSwedish,departureAirportEnglish,codeShareData,remarksEnglish,remarksSwedish,viaDestinations,diIndicator,airlineOperator.iata,airlineOperator.icao,...,baggage.baggageClaimUnit,arrivalTime.estimatedUtc,arrivalTime.actualUtc,locationAndStatus.gate,baggage.firstBagUtc,baggage.lastBagUtc,flightLegIdentifier.aircraftRegistration,flightLegIdentifier.ssrCode,airport,baggage.estimatedFirstBagUtc
0,FR915,Rom FCO,Rome FCO,[],[],[],[],S,FR,RYR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARN,NaN
1,FR4464,Birmingham,Birmingham,[],[],[],[],I,FR,RYR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARN,NaN
2,FR3008,Dublin,Dublin,[],[],[],[],I,FR,RYR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARN,NaN
3,FR7619,Palma,Palma,[],[],[],[],S,FR,RYR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARN,NaN
4,FR7694,Chania,Chania,[],[],[],[],S,FR,RYR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,SK1137,Stockholm ARN,Stockholm ARN,[KL8472],"[{'text': 'Landed 17:47', 'indicator': 'NEUTRA...","[{'text': 'Landat 17:47', 'indicator': 'NEUTRA...",[],D,SK,SAS,...,01,2026-10-07T15:54:00Z,2026-10-07T15:47:00Z,1,2026-10-07T16:01:12Z,2026-10-07T16:02:53Z,SEMKB,NaN,RNB,NaN
523,SK1127,Stockholm ARN,Stockholm ARN,[KL8462],"[{'text': 'Landed 19:49', 'indicator': 'NEUTRA...","[{'text': 'Landat 19:49', 'indicator': 'NEUTRA...",[],D,SK,SAS,...,01,2026-10-07T17:53:00Z,2026-10-07T17:49:00Z,1,2026-10-07T18:06:44Z,2026-10-07T18:07:39Z,SEMKI,NaN,RNB,NaN
524,SK1145,Stockholm ARN,Stockholm ARN,"[BT6173, KL8542]","[{'text': 'First bag on belt', 'indicator': 'N...","[{'text': 'Första bagage', 'indicator': 'NEUTR...",[],D,SK,SAS,...,01,2026-10-07T21:10:00Z,2026-10-07T21:05:00Z,1,2026-10-07T21:22:04Z,NaN,SEMKE,NaN,RNB,NaN
525,SK1042,Stockholm ARN,Stockholm ARN,"[EY3913, KL8500, LO5407]","[{'text': 'Landed 13:22', 'indicator': 'NEUTRA...","[{'text': 'Landat 13:22', 'indicator': 'NEUTRA...",[],D,SK,SAS,...,1,2026-10-07T11:21:00Z,2026-10-07T11:22:00Z,01,2026-10-07T11:37:01Z,2026-10-07T11:47:41Z,SEROE,NaN,KRN,NaN
